# Notebook 14 — TrOCR + Lexicon on the BD Benchmark

In [7]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"Using device: {DEVICE}")

Using device: mps


In [8]:
@dataclass
class Config:
    data_root: Path = Path("../data/doctors_prescription_bd")
    train_csv: str = "Training/training_labels.csv"; train_img_dir: str = "Training/training_words"
    val_csv: str = "Validation/validation_labels.csv"; val_img_dir: str = "Validation/validation_words"
    test_csv: str = "Testing/testing_labels.csv"; test_img_dir: str = "Testing/testing_words"
    img_col: str = "IMAGE"; label_col: str = "MEDICINE_NAME"
    processor_name: str = "microsoft/trocr-small-handwritten"
    batch_size: int = 8; epochs: int = 60; lr: float = 5e-5   # high ceiling; early stopping decides
    early_stop_patience: int = 8   # stop after 8 epochs with no val improvement
    max_target_len: int = 24; num_workers: int = 0
    ckpt_dir: Path = Path("../checkpoints/trocr_bd")

CFG = Config()
CFG.ckpt_dir.mkdir(parents=True, exist_ok=True)

## 1. Processor + model

In [9]:
processor = TrOCRProcessor.from_pretrained(CFG.processor_name)
model = VisionEncoderDecoderModel.from_pretrained(CFG.processor_name).to(DEVICE)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
print(f"TrOCR loaded | params={sum(p.numel() for p in model.parameters())/1e6:.0f}M")

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrOCR loaded | params=62M


## 2. BD dataset (labels lowercased, RGB images, tokenised targets)

In [10]:
class BDDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, processor):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col]).reset_index(drop=True)
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip().str.lower()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / str(row[self.cfg.img_col])).convert("RGB")
        pv = self.processor(img, return_tensors="pt").pixel_values.squeeze(0)
        labels = self.processor.tokenizer(row[self.cfg.label_col], padding="max_length",
                                          max_length=self.cfg.max_target_len, truncation=True).input_ids
        labels = [l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels]
        return {"pixel_values": pv, "labels": torch.tensor(labels),
                "text": row[self.cfg.label_col], "fname": str(row[self.cfg.img_col])}

def collate(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.stack([b["labels"] for b in batch]),
            "text": [b["text"] for b in batch], "fname": [b["fname"] for b in batch]}

train_ds = BDDataset(CFG.data_root / CFG.train_csv, CFG.data_root / CFG.train_img_dir, CFG, processor)
val_ds = BDDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.val_img_dir, CFG, processor)
test_ds = BDDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.test_img_dir, CFG, processor)
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

train_dl = DataLoader(train_ds, CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, collate_fn=collate)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

train=3120 val=780 test=780


## 3. Metrics

In [11]:
def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]

def corpus_metrics(preds, refs):
    tot = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    chars = sum(len(r) for r in refs); exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": tot / max(chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}

## 4. Fine-tune TrOCR on BD

In [12]:
@torch.no_grad()
def evaluate(loader):
    model.eval(); preds, refs, files = [], [], []
    for batch in loader:
        gen = model.generate(batch["pixel_values"].to(DEVICE), max_new_tokens=CFG.max_target_len)
        out = processor.batch_decode(gen, skip_special_tokens=True)
        preds += [o.strip().lower() for o in out]
        refs += [t.lower() for t in batch["text"]]; files += batch["fname"]
    return corpus_metrics(preds, refs), preds, refs, files

opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr)
# LR scheduler: halve LR when val CER plateaus, helps the model keep improving
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
best_cer, no_improve = float("inf"), 0
for epoch in range(1, CFG.epochs + 1):
    model.train(); t0, running = time.time(), 0.0
    for k, batch in enumerate(train_dl):
        out = model(pixel_values=batch["pixel_values"].to(DEVICE), labels=batch["labels"].to(DEVICE))
        loss = out.loss
        opt.zero_grad(); loss.backward(); opt.step()
        running += loss.item()
        if k % 20 == 0:
            print(f"  epoch {epoch} step {k}/{len(train_dl)} | loss {loss.item():.3f}", end="\r")
    vm, _, _, _ = evaluate(val_dl)
    sched.step(vm["CER"])
    print(f"\nepoch {epoch} | train loss {running/len(train_dl):.4f} | val CER {vm['CER']:.4f} | "
          f"val EM {vm['ExactMatch']:.4f} | {(time.time()-t0)/60:.1f} min")
    if vm["CER"] < best_cer:
        best_cer, no_improve = vm["CER"], 0
        model.save_pretrained(CFG.ckpt_dir / "best")
        print(f"  saved best (val CER {best_cer:.4f})")
    else:
        no_improve += 1
        if no_improve >= CFG.early_stop_patience:
            print(f"early stop at epoch {epoch} (no val improvement for {CFG.early_stop_patience} epochs)")
            break

  epoch 1 step 380/390 | loss 0.563
epoch 1 | train loss 1.4787 | val CER 0.1265 | val EM 0.7782 | 2.4 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.1265)
  epoch 2 step 380/390 | loss 0.032
epoch 2 | train loss 0.1887 | val CER 0.0656 | val EM 0.8872 | 2.5 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.0656)
  epoch 3 step 380/390 | loss 0.307
epoch 3 | train loss 0.1034 | val CER 0.1308 | val EM 0.8590 | 2.6 min
  epoch 4 step 380/390 | loss 0.068
epoch 4 | train loss 0.0712 | val CER 0.0599 | val EM 0.9128 | 2.5 min


...

  saved best (val CER 0.0261)
  epoch 12 step 380/390 | loss 0.000
epoch 12 | train loss 0.0004 | val CER 0.0219 | val EM 0.9692 | 2.4 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.0219)
  epoch 13 step 380/390 | loss 0.000
epoch 13 | train loss 0.0002 | val CER 0.0225 | val EM 0.9679 | 2.4 min
  epoch 14 step 380/390 | loss 0.000
epoch 14 | train loss 0.0001 | val CER 0.0219 | val EM 0.9679 | 2.4 min
  epoch 15 step 380/390 | loss 0.000
epoch 15 | train loss 0.0001 | val CER 0.0231 | val EM 0.9679 | 2.4 min
  epoch 16 step 380/390 | loss 0.000
epoch 16 | train loss 0.0003 | val CER 0.0198 | val EM 0.9718 | 2.4 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved best (val CER 0.0198)
  epoch 17 step 380/390 | loss 0.000
epoch 17 | train loss 0.0001 | val CER 0.0204 | val EM 0.9705 | 2.4 min
  epoch 18 step 380/390 | loss 0.000
epoch 18 | train loss 0.0001 | val CER 0.0237 | val EM 0.9667 | 2.4 min
  epoch 19 step 380/390 | loss 0.000
epoch 19 | train loss 0.0001 | val CER 0.0211 | val EM 0.9705 | 2.5 min
  epoch 20 step 380/390 | loss 0.000
epoch 20 | train loss 0.0001 | val CER 0.0206 | val EM 0.9705 | 2.5 min
  epoch 21 step 380/390 | loss 0.000
epoch 21 | train loss 0.0001 | val CER 0.0211 | val EM 0.9705 | 2.5 min
  epoch 22 step 380/390 | loss 0.000
epoch 22 | train loss 0.0001 | val CER 0.0221 | val EM 0.9705 | 2.5 min
  epoch 23 step 380/390 | loss 0.000
epoch 23 | train loss 0.0001 | val CER 0.0225 | val EM 0.9692 | 2.5 min
  epoch 24 step 380/390 | loss 0.000
epoch 24 | train loss 0.0001 | val CER 0.0219 | val EM 0.9692 | 2.5 min
early stop at epoch 24 (no val improvement for 8 epochs)


## 5. BD lexicon + decode, evaluate TrOCR vs TrOCR+lexicon

In [13]:
model = VisionEncoderDecoderModel.from_pretrained(CFG.ckpt_dir / "best").to(DEVICE).eval()

bd_lexicon = sorted(set(pd.read_csv(CFG.data_root / CFG.train_csv)[CFG.label_col]
                        .astype(str).str.strip().str.lower()))
by_len = defaultdict(list)
for w in bd_lexicon: by_len[len(w)].append(w)
print(f"BD lexicon size: {len(bd_lexicon)}")

def nearest(word, gap=3):
    if not word: return None, 10**9
    if word in by_len.get(len(word), ()): return word, 0
    best, bd = None, 10**9
    for L in range(len(word)-gap, len(word)+gap+1):
        for e in by_len.get(L, ()):
            d = edit_distance(word, e)
            if d < bd: best, bd = e, d
            if bd == 1: return best, bd
    return best, bd

def lexicon_decode(raw, tau):
    out = []
    for p in raw:
        e, d = nearest(p)
        out.append(e if (e is not None and d/max(len(p),1) <= tau) else p)
    return out

val_m, val_raw, val_refs, _ = evaluate(val_dl)
test_m, test_raw, test_refs, test_files = evaluate(test_dl)

best_tau, best_em = 0.0, -1
for tau in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    em = corpus_metrics(lexicon_decode(val_raw, tau), val_refs)["ExactMatch"]
    if em > best_em: best_em, best_tau = em, tau
print(f"selected τ={best_tau} (val EM {best_em:.4f})")

trocr_only = test_m
trocr_lex = corpus_metrics(lexicon_decode(test_raw, best_tau), test_refs)
print(f"\nBD TEST (n={trocr_only['n']}):")
print(f"  TrOCR            : CER {trocr_only['CER']:.4f} | EM {trocr_only['ExactMatch']:.4f}")
print(f"  TrOCR + lexicon  : CER {trocr_lex['CER']:.4f} | EM {trocr_lex['ExactMatch']:.4f}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

BD lexicon size: 78
selected τ=0.4 (val EM 0.9756)

BD TEST (n=780):
  TrOCR            : CER 0.0457 | EM 0.9410
  TrOCR + lexicon  : CER 0.0457 | EM 0.9410


## 6. Full BD comparison (CRNN vs TrOCR, both +lexicon)

In [14]:
print("""
BD BENCHMARK — FULL COMPARISON (test EM):
  CRNN baseline (M0)     : 0.636   (from Notebook 10)
  CRNN + lexicon (M2)    : 0.845   (from Notebook 10)
  TrOCR (fine-tuned)     : (above)
  TrOCR + lexicon        : (above)

Read against the custom-data result. If CRNN+lexicon (0.845) ~ TrOCR+lexicon on BD, the
finding is: on closed-vocabulary data the cheaper CRNN+lexicon matches the heavy TrOCR — i.e.
pretraining's advantage is specific to OPEN-vocabulary settings (where unseen names appear).
That is a precise, valuable conclusion about WHEN each approach is worth its cost.
""")

pd.DataFrame([{"dataset": "BD", "model": "TrOCR", **trocr_only},
              {"dataset": "BD", "model": "TrOCR+lexicon", **trocr_lex}]
             ).to_csv(CFG.ckpt_dir / "trocr_bd_results.csv", index=False)


BD BENCHMARK — FULL COMPARISON (test EM):
  CRNN baseline (M0)     : 0.636   (from Notebook 10)
  CRNN + lexicon (M2)    : 0.845   (from Notebook 10)
  TrOCR (fine-tuned)     : (above)
  TrOCR + lexicon        : (above)

Read against the custom-data result. If CRNN+lexicon (0.845) ~ TrOCR+lexicon on BD, the
finding is: on closed-vocabulary data the cheaper CRNN+lexicon matches the heavy TrOCR — i.e.
pretraining's advantage is specific to OPEN-vocabulary settings (where unseen names appear).
That is a precise, valuable conclusion about WHEN each approach is worth its cost.

